# NHÓM 6 – PHÂN TÍCH VÀ DỰ BÁO TRỄ HẠN GIAO HÀNG


## PHẦN 1. THIẾT LẬP MÔI TRƯỜNG VÀ ĐỌC DỮ LIỆU

### 1.1. Giới thiệu đề tài và bộ dữ liệu


**Đề tài:** Phân tích và dự báo trễ hạn giao hàng

**Bộ dữ liệu:** DataCo Smart Supply Chain Dataset

**Công nghệ:** Google Colab, Python, PySpark, Spark SQL, Spark MLlib và Plotly.

**Mục tiêu:**
- Phân tích tỷ lệ giao trễ theo khu vực.
- Phân tích theo phương thức vận chuyển.
- Phân tích tình trạng giao trễ theo thời gian.
- Phân tích sản phẩm và đặc điểm đơn hàng.
- Xây dựng mô hình dự báo nguy cơ giao trễ.
- Trực quan hóa kết quả bằng Dashboard Plotly tương tác và xuất file HTML.


### 1.2. Thiết lập môi trường Spark và kết nối Google Drive

Cài đặt PySpark, khởi tạo SparkSession và kết nối Google Drive để đọc dữ liệu, lưu kết quả xử lý và mô hình dự báo.

In [3]:
# Cài đặt PySpark
!pip -q install pyspark

from pyspark.sql import SparkSession
from google.colab import drive
import os

# Khởi tạo SparkSession
spark = (
    SparkSession.builder
    .appName("Nhom6_Delivery_Prediction")
    .getOrCreate()
)

# Kết nối Google Drive
mount_path = "/content/mydrive_new"

if not os.path.ismount(mount_path):
    drive.mount(mount_path)

# Khai báo thư mục dự án
base_path = f"{mount_path}/MyDrive/BigData_Delivery_Prediction"

raw_path = f"{base_path}/data_raw"
processed_path = f"{base_path}/data_processed"
models_path = f"{base_path}/models"
results_path = f"{base_path}/results"

# Tạo các thư mục nếu chưa tồn tại
for path in [raw_path, processed_path, models_path, results_path]:
    os.makedirs(path, exist_ok=True)

print("Spark version:", spark.version)
print("Google Drive:", base_path)
print("Thiết lập môi trường thành công!")

Mounted at /content/mydrive_new
Spark version: 4.0.4
Google Drive: /content/mydrive_new/MyDrive/BigData_Delivery_Prediction
Thiết lập môi trường thành công!


### 1.3. Đọc dữ liệu DataCo

Sử dụng PySpark để đọc bộ dữ liệu DataCo Smart Supply Chain từ Google Drive, kiểm tra số dòng, số cột và xem một số bản ghi mẫu.

In [ ]:
# Đường dẫn dữ liệu
data_path = f"{raw_path}/DataCoSupplyChainDataset.csv"

# Đọc dữ liệu bằng PySpark
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(data_path)
)

# Kiểm tra kích thước dữ liệu
print("Số dòng:", df_raw.count())
print("Số cột:", len(df_raw.columns))

# Hiển thị 5 bản ghi mẫu
df_raw.select(
    "Order Id",
    "Order Item Id",
    "Customer Id",
    "Shipping Mode",
    "Late_delivery_risk"
).show(5, truncate=False)

Số dòng: 180519
Số cột: 53
+--------+-------------+-----------+--------------+------------------+
|Order Id|Order Item Id|Customer Id|Shipping Mode |Late_delivery_risk|
+--------+-------------+-----------+--------------+------------------+
|77202   |180517       |20755      |Standard Class|0                 |
|75939   |179254       |19492      |Standard Class|1                 |
|75938   |179253       |19491      |Standard Class|0                 |
|75937   |179252       |19490      |Standard Class|0                 |
|75936   |179251       |19489      |Standard Class|0                 |
+--------+-------------+-----------+--------------+------------------+
only showing top 5 rows


### 1.4. Chuẩn hóa tên cột

Chuẩn hóa tên các cột về dạng chữ thường, không dấu cách và sử dụng dấu gạch dưới để thuận tiện cho việc xử lý dữ liệu bằng PySpark và Spark SQL.

In [ ]:
import re

# Chuẩn hóa tên cột
def normalize_column(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

df = df_raw.toDF(
    *[normalize_column(col) for col in df_raw.columns]
)

# Kiểm tra kết quả
print("Số cột:", len(df.columns))
print("Tên các cột sau chuẩn hóa:")
print(df.columns)

df.select(
    "order_id",
    "order_item_id",
    "customer_id",
    "shipping_mode",
    "late_delivery_risk"
).show(5, truncate=False)

Số cột: 53
Tên các cột sau chuẩn hóa:
['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date_dateorders', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_description', 'product_image', 'product_name', 'product_price', 'product_s

## PHẦN 2. KHÁM PHÁ VÀ TIỀN XỬ LÝ DỮ LIỆU



### 2.1. Kiểm tra cấu trúc và kiểu dữ liệu

Kiểm tra kích thước, cấu trúc và kiểu dữ liệu của bộ dữ liệu.

In [ ]:
print("Số dòng:", df.count())
print("Số cột:", len(df.columns))

df.printSchema()

Số dòng: 180519
Số cột: 53
root
 |-- type: string (nullable = true)
 |-- days_for_shipping_real: integer (nullable = true)
 |-- days_for_shipment_scheduled: integer (nullable = true)
 |-- benefit_per_order: double (nullable = true)
 |-- sales_per_customer: double (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- late_delivery_risk: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_fname: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_lname: string (nullable = true)
 |-- customer_password: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_street: string (nullable = true)
 |-- customer_zipcode: integer (nullable = true)
 |-- department_i

### 2.2. Kiểm tra giá trị thiếu và trùng lặp

Kiểm tra số lượng và tỷ lệ giá trị thiếu trong từng cột, đồng thời xác định các bản ghi trùng lặp và mã chi tiết đơn hàng trùng nhau để đánh giá chất lượng dữ liệu trước khi làm sạch.

In [ ]:
from pyspark.sql import functions as F

# Tổng số bản ghi
total_rows = df.count()

# 1. Kiểm tra giá trị thiếu
missing_exprs = [
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
]

missing_counts = df.agg(*missing_exprs).first().asDict()

missing_data = [
    (col, int(count), round(count / total_rows * 100, 2))
    for col, count in missing_counts.items()
    if count > 0
]

df_missing = spark.createDataFrame(
    missing_data,
    ["column", "missing_count", "missing_percent"]
)

print("===== GIÁ TRỊ THIẾU =====")
df_missing.orderBy(
    F.desc("missing_percent")
).show(truncate=False)

# 2. Kiểm tra trùng lặp
duplicate_rows = total_rows - df.dropDuplicates().count()

duplicate_items = (
    total_rows - df.select("order_item_id").distinct().count()
)

print("===== TRÙNG LẶP =====")
print("Tổng số dòng:", total_rows)
print("Số dòng trùng hoàn toàn:", duplicate_rows)
print("Số mã Order Item trùng:", duplicate_items)

===== GIÁ TRỊ THIẾU =====
+-------------------+-------------+---------------+
|column             |missing_count|missing_percent|
+-------------------+-------------+---------------+
|product_description|180519       |100.0          |
|order_zipcode      |155679       |86.24          |
|customer_lname     |8            |0.0            |
|customer_zipcode   |3            |0.0            |
+-------------------+-------------+---------------+

===== TRÙNG LẶP =====
Tổng số dòng: 180519
Số dòng trùng hoàn toàn: 0
Số mã Order Item trùng: 0


### 2.3. Làm sạch và kiểm tra tính hợp lệ của dữ liệu

Loại bỏ các cột có tỷ lệ thiếu dữ liệu lớn, chuyển đổi các cột ngày tháng về kiểu Timestamp và kiểm tra tính hợp lệ của dữ liệu sau xử lý.

Dữ liệu được làm sạch nhưng vẫn giữ nguyên số bản ghi để phục vụ các bước thiết kế bảng, phân tích và dự báo.

In [ ]:
from pyspark.sql import functions as F

# 1. Loại bỏ các cột có tỷ lệ thiếu lớn
df_clean = df.drop(
    "product_description",
    "order_zipcode"
)

# 2. Chuyển đổi kiểu dữ liệu ngày tháng
date_columns = [
    "order_date_dateorders",
    "shipping_date_dateorders"
]

for c in date_columns:
    df_clean = df_clean.withColumn(
        c,
        F.to_timestamp(F.col(c), "M/d/yyyy H:mm")
    )

# 3. Kiểm tra tính hợp lệ
validation = df_clean.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.col("order_item_id").isNull().cast("int")
    ).alias("null_order_item_id"),

    F.sum(
        F.col("order_date_dateorders").isNull().cast("int")
    ).alias("null_order_date"),

    F.sum(
        F.col("shipping_date_dateorders").isNull().cast("int")
    ).alias("null_shipping_date"),

    F.sum(
        (
            F.col("shipping_date_dateorders")
            < F.col("order_date_dateorders")
        ).cast("int")
    ).alias("invalid_date_order"),

    F.sum(
        (
            (F.col("days_for_shipping_real") < 0) |
            (F.col("days_for_shipment_scheduled") < 0)
        ).cast("int")
    ).alias("invalid_shipping_days")
)

# 4. Hiển thị kết quả
print("===== KẾT QUẢ LÀM SẠCH =====")
print("Số dòng:", df_clean.count())
print("Số cột:", len(df_clean.columns))

print("\n===== KIỂM TRA TÍNH HỢP LỆ =====")
validation.show(truncate=False)

print("\n===== PHÂN BỐ NHÃN GIAO TRỄ =====")
df_clean.groupBy("late_delivery_risk") \
    .count() \
    .orderBy("late_delivery_risk") \
    .show()

===== KẾT QUẢ LÀM SẠCH =====
Số dòng: 180519
Số cột: 51

===== KIỂM TRA TÍNH HỢP LỆ =====
+----------+------------------+---------------+------------------+------------------+---------------------+
|total_rows|null_order_item_id|null_order_date|null_shipping_date|invalid_date_order|invalid_shipping_days|
+----------+------------------+---------------+------------------+------------------+---------------------+
|180519    |0                 |0              |0                 |0                 |0                    |
+----------+------------------+---------------+------------------+------------------+---------------------+


===== PHÂN BỐ NHÃN GIAO TRỄ =====
+------------------+-----+
|late_delivery_risk|count|
+------------------+-----+
|                 0|81542|
|                 1|98977|
+------------------+-----+



## PHẦN 3. THIẾT KẾ VÀ TÍCH HỢP DỮ LIỆU



### 3.1. Thiết kế 6 bảng dữ liệu

Tách dữ liệu đã làm sạch thành 6 bảng: Customers, Products,
Orders, OrderItems, Shipping và Regions.

Mỗi bảng lưu trữ thông tin của một thực thể, sử dụng khóa chính
để định danh và khóa ngoại để liên kết dữ liệu.

In [ ]:
from pyspark.sql import functions as F

# 1. Customers
df_customers = (
    df_clean.select(
        "customer_id",
        "customer_segment",
        "customer_city",
        "customer_state",
        "customer_country"
    )
    .dropDuplicates(["customer_id"])
)

# 2. Products
df_products = (
    df_clean.select(
        "product_card_id",
        "product_name",
        "product_category_id"
    )
    .dropDuplicates(["product_card_id"])
)

# 3. Orders
df_orders = (
    df_clean.select(
        "order_id",
        "customer_id",
        "order_date_dateorders",
        "order_region",
        "order_country"
    )
    .dropDuplicates(["order_id"])
)

# 4. OrderItems
df_order_items = (
    df_clean.select(
        "order_item_id",
        "order_id",
        "product_card_id",
        "order_item_quantity",
        "order_item_total"
    )
    .dropDuplicates(["order_item_id"])
)

# 5. Shipping
df_shipping = (
    df_clean.select(
        "order_id",
        "shipping_mode",
        "days_for_shipment_scheduled",
        "days_for_shipping_real",
        "shipping_date_dateorders",
        "late_delivery_risk"
    )
    .dropDuplicates(["order_id"])
)

# 6. Regions
df_regions = (
    df_clean.select(
        "order_region",
        "order_country"
    )
    .dropDuplicates()
)

# Kiểm tra kết quả
tables = {
    "Customers": df_customers,
    "Products": df_products,
    "Orders": df_orders,
    "OrderItems": df_order_items,
    "Shipping": df_shipping,
    "Regions": df_regions
}

for name, table in tables.items():
    print(f"{name}: {table.count()} dòng, {len(table.columns)} cột")

Customers: 20652 dòng, 5 cột
Products: 118 dòng, 3 cột
Orders: 65752 dòng, 5 cột
OrderItems: 180519 dòng, 5 cột
Shipping: 65752 dòng, 6 cột
Regions: 167 dòng, 2 cột


### 3.2. Kiểm tra khóa chính và khóa ngoại

Kiểm tra tính duy nhất, không rỗng của khóa chính và tính toàn vẹn tham chiếu của khóa ngoại giữa các bảng nhằm đảm bảo dữ liệu nhất quán trước khi tích hợp.

In [ ]:
from pyspark.sql import functions as F

# Kiểm tra khóa chính
primary_keys = [
    ("Customers", df_customers, ["customer_id"]),
    ("Products", df_products, ["product_card_id"]),
    ("Orders", df_orders, ["order_id"]),
    ("OrderItems", df_order_items, ["order_item_id"]),
    ("Shipping", df_shipping, ["order_id"]),
    ("Regions", df_regions, ["order_region", "order_country"])
]

print("===== KIỂM TRA KHÓA CHÍNH =====")

for name, table, keys in primary_keys:
    total = table.count()
    unique = table.select(*keys).distinct().count()

    null_count = table.filter(
        " OR ".join(f"`{key}` IS NULL" for key in keys)
    ).count()

    status = "ĐẠT" if total == unique and null_count == 0 else "KHÔNG ĐẠT"

    print(f"{name}: {status} | Trùng: {total - unique} | NULL: {null_count}")


# Kiểm tra khóa ngoại
foreign_keys = [
    ("Orders → Customers", df_orders, df_customers, ["customer_id"]),
    ("OrderItems → Orders", df_order_items, df_orders, ["order_id"]),
    ("OrderItems → Products", df_order_items, df_products, ["product_card_id"]),
    ("Shipping → Orders", df_shipping, df_orders, ["order_id"]),
    ("Orders → Regions", df_orders, df_regions, ["order_region", "order_country"])
]

print("\n===== KIỂM TRA KHÓA NGOẠI =====")

for name, child, parent, keys in foreign_keys:
    invalid = (
        child.join(
            parent.select(*keys).distinct(),
            on=keys,
            how="left_anti"
        ).count()
    )

    status = "ĐẠT" if invalid == 0 else "KHÔNG ĐẠT"
    print(f"{name}: {status} | Không khớp: {invalid}")

===== KIỂM TRA KHÓA CHÍNH =====
Customers: ĐẠT | Trùng: 0 | NULL: 0
Products: ĐẠT | Trùng: 0 | NULL: 0
Orders: ĐẠT | Trùng: 0 | NULL: 0
OrderItems: ĐẠT | Trùng: 0 | NULL: 0
Shipping: ĐẠT | Trùng: 0 | NULL: 0
Regions: ĐẠT | Trùng: 0 | NULL: 0

===== KIỂM TRA KHÓA NGOẠI =====
Orders → Customers: ĐẠT | Không khớp: 0
OrderItems → Orders: ĐẠT | Không khớp: 0
OrderItems → Products: ĐẠT | Không khớp: 0
Shipping → Orders: ĐẠT | Không khớp: 0
Orders → Regions: ĐẠT | Không khớp: 0


### 3.3. Tích hợp dữ liệu bằng Spark JOIN

Sử dụng PySpark để kết hợp 6 bảng dữ liệu theo các quan hệ khóa chính – khóa ngoại, tạo bảng dữ liệu tích hợp phục vụ phân tích bằng Spark SQL và xây dựng mô hình dự báo.

Kiểm tra số lượng bản ghi, số cột và tính duy nhất của mã chi tiết đơn hàng sau khi tích hợp.

In [ ]:
from pyspark.sql import functions as F

# Tích hợp 6 bảng dữ liệu
df_integrated = (
    df_order_items
    .join(df_orders, "order_id", "inner")
    .join(df_customers, "customer_id", "inner")
    .join(df_products, "product_card_id", "inner")
    .join(df_shipping, "order_id", "inner")
    .join(
        df_regions,
        ["order_region", "order_country"],
        "inner"
    )
)

# Kiểm tra kết quả tích hợp
total_rows = df_integrated.count()
total_columns = len(df_integrated.columns)

duplicate_items = (
    total_rows
    - df_integrated.select("order_item_id").distinct().count()
)

missing_items = (
    df_order_items.select("order_item_id")
    .join(
        df_integrated.select("order_item_id"),
        "order_item_id",
        "left_anti"
    )
    .count()
)

print("===== KẾT QUẢ TÍCH HỢP =====")
print("Số dòng:", total_rows)
print("Số cột:", total_columns)
print("Mã chi tiết đơn hàng trùng:", duplicate_items)
print("Mã chi tiết đơn hàng bị mất:", missing_items)

# Hiển thị dữ liệu mẫu
df_integrated.select(
    "order_id",
    "order_item_id",
    "customer_id",
    "product_card_id",
    "order_region",
    "shipping_mode",
    "late_delivery_risk"
).show(5, truncate=False)

===== KẾT QUẢ TÍCH HỢP =====
Số dòng: 180519
Số cột: 20
Mã chi tiết đơn hàng trùng: 0
Mã chi tiết đơn hàng bị mất: 0
+--------+-------------+-----------+---------------+---------------+--------------+------------------+
|order_id|order_item_id|customer_id|product_card_id|order_region   |shipping_mode |late_delivery_risk|
+--------+-------------+-----------+---------------+---------------+--------------+------------------+
|75902   |179217       |19455      |1360           |Oceania        |Same Day      |1                 |
|58034   |145203       |6277       |191            |Central America|Standard Class|1                 |
|75800   |179115       |19353      |1360           |South Asia     |Standard Class|1                 |
|48374   |120988       |1834       |191            |Central Africa |Second Class  |1                 |
|11494   |28759        |4366       |403            |Northern Europe|Standard Class|1                 |
+--------+-------------+-----------+---------------+-------

### 3.4. Lưu dữ liệu Parquet và đăng ký Spark SQL View

Lưu 6 bảng dữ liệu và bảng tích hợp dưới định dạng Parquet trên Google Drive nhằm hỗ trợ lưu trữ và tái sử dụng dữ liệu.

Đồng thời đăng ký các bảng dưới dạng Temporary View để thực hiện truy vấn và phân tích bằng Spark SQL ở Phần 4.

In [ ]:
# Danh sách 7 bảng dữ liệu
parquet_tables = {
    "customers": df_customers,
    "products": df_products,
    "orders": df_orders,
    "order_items": df_order_items,
    "shipping": df_shipping,
    "regions": df_regions,
    "integrated_data": df_integrated
}

# Lưu Parquet và đăng ký Spark SQL View
for name, table in parquet_tables.items():

    # Lưu dữ liệu
    table.write.mode("overwrite").parquet(
        f"{processed_path}/{name}.parquet"
    )

    # Đăng ký Temporary View
    table.createOrReplaceTempView(name)

    print(f"{name}: {table.count()} dòng - Đã lưu và đăng ký View")


customers: 20652 dòng - Đã lưu và đăng ký View
products: 118 dòng - Đã lưu và đăng ký View
orders: 65752 dòng - Đã lưu và đăng ký View
order_items: 180519 dòng - Đã lưu và đăng ký View
shipping: 65752 dòng - Đã lưu và đăng ký View
regions: 167 dòng - Đã lưu và đăng ký View
integrated_data: 180519 dòng - Đã lưu và đăng ký View


**Kiến trúc lưu trữ dữ liệu**

Hệ thống sử dụng Google Drive làm nơi lưu trữ các tệp dữ liệu,
được tổ chức thành các lớp:

- Raw Data: Lưu dữ liệu DataCo gốc ở định dạng CSV.
- Processed Data: Lưu 6 bảng dữ liệu chuẩn hóa và 1 bảng
  dữ liệu tích hợp dưới định dạng Parquet.
- Analytical Data: Lưu các kết quả phân tích bằng Spark SQL và đánh giá mô hình dưới định dạng CSV để phục vụ trực quan hóa bằng Plotly và xây dựng Dashboard HTML tương tác.
  dưới định dạng CSV để phục vụ Power BI.
- Model Storage: Lưu Pipeline Random Forest đã huấn luyện.

Các bảng Parquet được đăng ký thành Spark SQL Temporary Views
để phục vụ truy vấn và phân tích dữ liệu.

Kiến trúc này mô phỏng cách tổ chức dữ liệu theo các lớp
Raw – Processed – Analytical, chưa triển khai một hệ thống
Data Warehouse chuyên dụng.

## PHẦN 4. PHÂN TÍCH DỮ LIỆU BẰNG SPARK SQL



### 4.1. Tổng quan tình trạng giao hàng

Phân tích số lượng đơn hàng, số đơn giao trễ
và tỷ lệ giao trễ trên toàn bộ dữ liệu.

In [ ]:
df_overview = spark.sql("""
    SELECT
        COUNT(*) AS total_orders,
        SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) AS late_orders,
        SUM(CASE WHEN late_delivery_risk = 0 THEN 1 ELSE 0 END) AS on_time_orders,
        ROUND(AVG(late_delivery_risk) * 100, 2) AS late_rate_percent
    FROM shipping
""")

df_overview.show()

+------------+-----------+--------------+-----------------+
|total_orders|late_orders|on_time_orders|late_rate_percent|
+------------+-----------+--------------+-----------------+
|       65752|      36048|         29704|            54.82|
+------------+-----------+--------------+-----------------+



### 4.2. Phân tích giao trễ theo khu vực

Thống kê tổng số đơn hàng, số đơn giao trễ và tỷ lệ giao trễ theo từng khu vực nhằm xác định những khu vực có nguy cơ giao hàng trễ cao.

In [ ]:
df_region_analysis = spark.sql("""
    SELECT
        o.order_region,
        COUNT(*) AS total_orders,
        SUM(s.late_delivery_risk) AS late_orders,
        ROUND(AVG(s.late_delivery_risk) * 100, 2)
            AS late_rate_percent
    FROM orders o
    JOIN shipping s ON o.order_id = s.order_id
    GROUP BY o.order_region
    ORDER BY late_rate_percent DESC
""")

df_region_analysis.show(23, truncate=False)

+---------------+------------+-----------+-----------------+
|order_region   |total_orders|late_orders|late_rate_percent|
+---------------+------------+-----------+-----------------+
|Central Africa |556         |320        |57.55            |
|East Africa    |613         |348        |56.77            |
|South of  USA  |1345        |753        |55.99            |
|West Asia      |2022        |1132       |55.98            |
|Eastern Europe |1292        |723        |55.96            |
|South Asia     |3335        |1865       |55.92            |
|Western Europe |10010       |5585       |55.79            |
|Southeast Asia |4356        |2421       |55.58            |
|Central Asia   |184         |102        |55.43            |
|US Center      |1935        |1071       |55.35            |
|East of USA    |2323        |1276       |54.93            |
|Central America|9396        |5156       |54.87            |
|Eastern Asia   |3318        |1818       |54.79            |
|Oceania        |4362   

### 4.3. Phân tích giao trễ theo phương thức vận chuyển

Thống kê số lượng đơn hàng và tỷ lệ giao trễ theo từng phương thức vận chuyển nhằm đánh giá sự khác biệt về tình trạng giao hàng giữa các phương thức.

In [ ]:
df_shipping_analysis = spark.sql("""
    SELECT
        shipping_mode,
        COUNT(*) AS total_orders,
        SUM(late_delivery_risk) AS late_orders,
        ROUND(AVG(late_delivery_risk) * 100, 2)
            AS late_rate_percent
    FROM shipping
    GROUP BY shipping_mode
    ORDER BY late_rate_percent DESC
""")

df_shipping_analysis.show(truncate=False)

+--------------+------------+-----------+-----------------+
|shipping_mode |total_orders|late_orders|late_rate_percent|
+--------------+------------+-----------+-----------------+
|First Class   |10079       |9602       |95.27            |
|Second Class  |12778       |9803       |76.72            |
|Same Day      |3571        |1648       |46.15            |
|Standard Class|39324       |14995      |38.13            |
+--------------+------------+-----------+-----------------+



### 4.4. Phân tích giao trễ theo thời gian

Thống kê tổng số đơn hàng, số đơn giao trễ và tỷ lệ giao trễ theo từng tháng, năm nhằm theo dõi sự biến động của tình trạng giao hàng trễ theo thời gian.

In [ ]:
df_time_analysis = spark.sql("""
    SELECT
        YEAR(o.order_date_dateorders) AS order_year,
        MONTH(o.order_date_dateorders) AS order_month,
        COUNT(*) AS total_orders,
        SUM(s.late_delivery_risk) AS late_orders,
        ROUND(AVG(s.late_delivery_risk) * 100, 2)
            AS late_rate_percent
    FROM orders o
    JOIN shipping s ON o.order_id = s.order_id
    GROUP BY
        YEAR(o.order_date_dateorders),
        MONTH(o.order_date_dateorders)
    ORDER BY order_year, order_month
""")

df_time_analysis.show(40, truncate=False)

+----------+-----------+------------+-----------+-----------------+
|order_year|order_month|total_orders|late_orders|late_rate_percent|
+----------+-----------+------------+-----------+-----------------+
|2015      |1          |1787        |963        |53.89            |
|2015      |2          |1585        |871        |54.95            |
|2015      |3          |1781        |978        |54.91            |
|2015      |4          |1710        |927        |54.21            |
|2015      |5          |1776        |969        |54.56            |
|2015      |6          |1725        |941        |54.55            |
|2015      |7          |1763        |963        |54.62            |
|2015      |8          |1762        |966        |54.82            |
|2015      |9          |1706        |951        |55.74            |
|2015      |10         |1775        |973        |54.82            |
|2015      |11         |1729        |941        |54.42            |
|2015      |12         |1805        |988        

### 4.5. Phân tích giao trễ theo sản phẩm và đặc điểm đơn hàng

Phân tích tình trạng giao trễ theo danh mục sản phẩm và quy mô đơn hàng nhằm đánh giá mối liên hệ giữa đặc điểm đơn hàng với tỷ lệ giao trễ.

In [ ]:
# 1. Phân tích theo danh mục sản phẩm
df_product_analysis = spark.sql("""
    SELECT
        p.product_category_id,
        COUNT(DISTINCT oi.order_id) AS total_orders,
        COUNT(DISTINCT CASE
            WHEN s.late_delivery_risk = 1 THEN oi.order_id
        END) AS late_orders,
        ROUND(
            COUNT(DISTINCT CASE
                WHEN s.late_delivery_risk = 1 THEN oi.order_id
            END) * 100.0
            / COUNT(DISTINCT oi.order_id), 2
        ) AS late_rate_percent
    FROM order_items oi
    JOIN products p ON oi.product_card_id = p.product_card_id
    JOIN shipping s ON oi.order_id = s.order_id
    GROUP BY p.product_category_id
    ORDER BY total_orders DESC
""")

df_product_analysis.show(20, truncate=False)


# 2. Phân tích theo quy mô đơn hàng
df_order_size_analysis = spark.sql("""
    WITH order_size AS (
        SELECT
            order_id,
            SUM(order_item_quantity) AS total_quantity,
            ROUND(SUM(order_item_total), 2) AS total_value
        FROM order_items
        GROUP BY order_id
    )
    SELECT
        CASE
            WHEN os.total_quantity = 1 THEN '1 sản phẩm'
            WHEN os.total_quantity BETWEEN 2 AND 3 THEN '2-3 sản phẩm'
            ELSE '4+ sản phẩm'
        END AS quantity_group,
        COUNT(*) AS total_orders,
        SUM(s.late_delivery_risk) AS late_orders,
        ROUND(AVG(s.late_delivery_risk) * 100, 2)
            AS late_rate_percent,
        ROUND(AVG(os.total_value), 2) AS avg_order_value
    FROM order_size os
    JOIN shipping s ON os.order_id = s.order_id
    GROUP BY
        CASE
            WHEN os.total_quantity = 1 THEN '1 sản phẩm'
            WHEN os.total_quantity BETWEEN 2 AND 3 THEN '2-3 sản phẩm'
            ELSE '4+ sản phẩm'
        END
    ORDER BY quantity_group
""")

df_order_size_analysis.show(truncate=False)

+-------------------+------------+-----------+-----------------+
|product_category_id|total_orders|late_orders|late_rate_percent|
+-------------------+------------+-----------+-----------------+
|17                 |20386       |11228      |55.08            |
|18                 |18783       |10247      |54.55            |
|24                 |17869       |9723       |54.41            |
|46                 |16623       |9113       |54.82            |
|45                 |15164       |8330       |54.93            |
|48                 |13758       |7568       |55.01            |
|43                 |12299       |6719       |54.63            |
|9                  |11355       |6206       |54.65            |
|29                 |10136       |5582       |55.07            |
|37                 |1994        |1122       |56.27            |
|40                 |1753        |999        |56.99            |
|36                 |1444        |787        |54.50            |
|26                 |1191

## PHẦN 5. XÂY DỰNG MÔ HÌNH DỰ BÁO GIAO TRỄ



### 5.1. Chuẩn bị dữ liệu dự báo

Xây dựng dữ liệu cấp đơn hàng, kết hợp đặc trưng
khách hàng, vận chuyển, thời gian và quy mô đơn hàng.

Nhãn dự báo là late_delivery_risk.

In [ ]:
from pyspark.sql import functions as F

# Tổng hợp đặc trưng từ chi tiết đơn hàng
df_order_features = (
    df_order_items
    .groupBy("order_id")
    .agg(
        F.count("*").alias("num_order_items"),
        F.sum("order_item_quantity").alias("total_quantity"),
        F.round(F.sum("order_item_total"), 2).alias("total_order_value"),
        F.countDistinct("product_card_id").alias("num_distinct_products")
    )
)

# Tạo dữ liệu dự báo ở cấp đơn hàng
df_model = (
    df_orders
    .join(df_customers, "customer_id", "inner")
    .join(
        df_shipping.select(
            "order_id",
            "shipping_mode",
            "days_for_shipment_scheduled",
            "late_delivery_risk"
        ),
        "order_id",
        "inner"
    )
    .join(df_order_features, "order_id", "inner")
    .select(
        "order_id",
        F.col("order_date_dateorders").alias("order_date"),
        "order_region",
        "order_country",
        "customer_segment",
        "shipping_mode",
        F.col("days_for_shipment_scheduled").alias("scheduled_shipping_days"),
        "num_order_items",
        "total_quantity",
        "total_order_value",
        "num_distinct_products",
        F.col("late_delivery_risk").cast("double").alias("label")
    )
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("order_day_of_week", F.dayofweek("order_date"))
)

# Kiểm tra dữ liệu
print("===== DỮ LIỆU MÔ HÌNH =====")
print("Số đơn hàng:", df_model.count())
print("Số cột:", len(df_model.columns))
print(
    "Số Order ID trùng:",
    df_model.count() - df_model.select("order_id").distinct().count()
)

df_model.groupBy("label").count().orderBy("label").show()

===== DỮ LIỆU MÔ HÌNH =====
Số đơn hàng: 65752
Số cột: 15
Số Order ID trùng: 0
+-----+-----+
|label|count|
+-----+-----+
|  0.0|29704|
|  1.0|36048|
+-----+-----+



### 5.2. Lựa chọn đặc trưng và chia dữ liệu

Sử dụng các đặc trưng có thể biết tại thời điểm đặt hàng.

Chia dữ liệu theo thời gian thành Train, Validation và Test
để hạn chế rò rỉ thông tin từ tương lai.

In [ ]:
from pyspark.sql import functions as F

# Đặc trưng phân loại
categorical_cols = [
    "order_region",
    "order_country",
    "customer_segment",
    "shipping_mode"
]

# Đặc trưng số
numeric_cols = [
    "scheduled_shipping_days",
    "order_year",
    "order_month",
    "order_day_of_week",
    "num_order_items",
    "total_quantity",
    "total_order_value",
    "num_distinct_products"
]

print("Số đặc trưng đầu vào:",
      len(categorical_cols) + len(numeric_cols))

# Chia dữ liệu theo thời gian
df_train = df_model.filter(
    F.col("order_date") < "2017-01-01"
)

df_val = df_model.filter(
    (F.col("order_date") >= "2017-01-01") &
    (F.col("order_date") < "2017-07-01")
)

df_test = df_model.filter(
    F.col("order_date") >= "2017-07-01"
)

print("Train:", df_train.count())
print("Validation:", df_val.count())
print("Test:", df_test.count())

Số đặc trưng đầu vào: 12
Train: 41763
Validation: 10319
Test: 13670


Nhóm sử dụng 12 đặc trưng có thể biết tại thời điểm đặt hàng và chia dữ liệu theo thời gian thành Train, Validation và Test. Cách chia này giúp đánh giá mô hình trên các đơn hàng phát sinh sau giai đoạn huấn luyện, đồng thời hạn chế rò rỉ dữ liệu.

### 5.3. Xây dựng Pipeline xử lý đặc trưng

Sử dụng PySpark MLlib để xây dựng Pipeline tiền xử lý dữ liệu.

Các đặc trưng phân loại được chuyển thành dạng số bằng StringIndexer và OneHotEncoder. Sau đó, VectorAssembler kết hợp toàn bộ 12 đặc trưng thành một vector đầu vào cho mô hình.

Pipeline giúp đảm bảo quá trình xử lý dữ liệu nhất quán giữa các tập Train, Validation và Test.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

# Mã hóa đặc trưng phân loại
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_index",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_index",
        outputCol=f"{c}_vec",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

# Kết hợp đặc trưng
assembler = VectorAssembler(
    inputCols=[f"{c}_vec" for c in categorical_cols] + numeric_cols,
    outputCol="features",
    handleInvalid="keep"
)

# Pipeline tiền xử lý
preprocessing_pipeline = Pipeline(
    stages=indexers + encoders + [assembler]
)

print("Số đặc trưng gốc:",
      len(categorical_cols) + len(numeric_cols))

print("Số bước trong Pipeline:",
      len(preprocessing_pipeline.getStages()))

Số đặc trưng gốc: 12
Số bước trong Pipeline: 9


### 5.4. Huấn luyện mô hình

Sử dụng PySpark MLlib để huấn luyện hai thuật toán phân loại:
Logistic Regression và Random Forest.

Cả hai mô hình sử dụng cùng 12 đặc trưng đầu vào, cùng Pipeline
tiền xử lý và cùng tập Train để đảm bảo tính công bằng khi so sánh.

Tập Validation được sử dụng để lựa chọn mô hình phù hợp.
Tập Test được giữ riêng để đánh giá cuối cùng.

In [ ]:
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier
)

# Logistic Regression
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01
)

pipeline_lr = Pipeline(
    stages=preprocessing_pipeline.getStages() + [lr]
)

# Random Forest
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=8,
    seed=42
)

pipeline_rf = Pipeline(
    stages=preprocessing_pipeline.getStages() + [rf]
)

# Huấn luyện trên tập Train
model_lr = pipeline_lr.fit(df_train)
model_rf = pipeline_rf.fit(df_train)

print("Đã huấn luyện Logistic Regression")
print("Đã huấn luyện Random Forest")

Đã huấn luyện Logistic Regression
Đã huấn luyện Random Forest


In [ ]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    maxDepth=5,
    seed=42
)

pipeline_gbt = Pipeline(
    stages=preprocessing_pipeline.getStages() + [gbt]
)

model_gbt = pipeline_gbt.fit(df_train)

print("Đã huấn luyện Gradient-Boosted Trees")

Đã huấn luyện Gradient-Boosted Trees


### 5.5. Đánh giá và lựa chọn mô hình

Đánh giá hai mô hình trên tập Validation bằng Accuracy,
Precision, Recall, F1-score và ROC-AUC.

Lựa chọn mô hình phù hợp trước khi đánh giá trên Test.

In [ ]:
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)
from pyspark.sql import functions as F

# Dự báo trên tập Validation
pred_lr_val = model_lr.transform(df_val)
pred_rf_val = model_rf.transform(df_val)
pred_gbt_val = model_gbt.transform(df_val)

# Hàm đánh giá mô hình
def evaluate_model(pred, model_name):
    metrics = {}

    for metric in ["accuracy", "weightedPrecision",
                   "weightedRecall", "f1"]:
        evaluator = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName=metric
        )
        metrics[metric] = round(evaluator.evaluate(pred), 4)

    auc_evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    metrics["roc_auc"] = round(auc_evaluator.evaluate(pred), 4)

    return (
        model_name,
        metrics["accuracy"],
        metrics["weightedPrecision"],
        metrics["weightedRecall"],
        metrics["f1"],
        metrics["roc_auc"]
    )

# So sánh 3 mô hình
results = [
    evaluate_model(pred_lr_val, "Logistic Regression"),
    evaluate_model(pred_rf_val, "Random Forest"),
    evaluate_model(pred_gbt_val, "Gradient-Boosted Trees")
]

df_model_comparison = spark.createDataFrame(
    results,
    ["model", "accuracy", "precision",
     "recall", "f1", "roc_auc"]
)


# Xếp hạng theo Weighted F1-score (tiêu chí lựa chọn mô hình)
df_model_comparison.orderBy(
    F.desc("f1")
).show(truncate=False)


+----------------------+--------+---------+------+------+-------+
|model                 |accuracy|precision|recall|f1    |roc_auc|
+----------------------+--------+---------+------+------+-------+
|Random Forest         |0.6889  |0.7138   |0.6889|0.6861|0.7244 |
|Logistic Regression   |0.6897  |0.7326   |0.6897|0.683 |0.7254 |
|Gradient-Boosted Trees|0.6892  |0.7313   |0.6892|0.6827|0.7257 |
+----------------------+--------+---------+------+------+-------+



### 5.6. Đánh giá mô hình Random Forest trên tập Test

In [ ]:
# Dự đoán trên Test
pred_rf_test = model_rf.transform(df_test)

# Đánh giá Random Forest
result_test = evaluate_model(
    pred_rf_test,
    "Random Forest - Test"
)

print("Kết quả trên Test:")
for name, value in zip(
    ["Model", "Accuracy", "Precision",
     "Recall", "Weighted F1", "ROC-AUC"],
    result_test
):
    print(f"{name}: {value}")

Kết quả trên Test:
Model: Random Forest - Test
Accuracy: 0.6948
Precision: 0.721
Recall: 0.6948
Weighted F1: 0.6929
ROC-AUC: 0.7309


Mô hình Random Forest đạt Accuracy 69,48%, Weighted F1-score 69,29% và ROC-AUC 73,09% trên tập Test. Kết quả cho thấy mô hình có khả năng dự báo giao hàng trễ ở mức tương đối, tuy nhiên vẫn còn sai sót.

## PHẦN 6. THỰC NGHIỆM MỞ RỘNG VÀ LỰA CHỌN ĐẶC TRƯNG CHO MÔ HÌNH DỰ BÁO

### 6.1. Mục tiêu cải tiến mô hình

Rút gọn từ 12 xuống 5 đặc trưng và bổ sung 2 đặc trưng về tỷ lệ giao trễ lịch sử theo phương thức vận chuyển và khu vực nhằm cải thiện độ chính xác của mô hình Random Forest.

### 6.2. Lựa chọn 5 đặc trưng

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

cat_cols = ["shipping_mode", "order_region", "order_country"]
num_cols = ["scheduled_shipping_days", "order_month"]

def train_rf(train, val, numeric_cols):
    indexers = [
        StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
        for c in cat_cols
    ]

    encoder = OneHotEncoder(
        inputCols=[c+"_idx" for c in cat_cols],
        outputCols=[c+"_ohe" for c in cat_cols]
    )

    assembler = VectorAssembler(
        inputCols=[c+"_ohe" for c in cat_cols] + numeric_cols,
        outputCol="features"
    )

    rf = RandomForestClassifier(
        labelCol="label",
        featuresCol="features",
        numTrees=100,
        maxDepth=8,
        seed=42
    )

    model = Pipeline(stages=indexers + [encoder, assembler, rf]).fit(train)
    return model, evaluate_model(model.transform(val), "Random Forest")

In [ ]:
model_5, result_5 = train_rf(df_train, df_val, num_cols)

print("Mô hình 5 đặc trưng:")
print(result_5)

Mô hình 5 đặc trưng:
('Random Forest', 0.6889, 0.7138, 0.6889, 0.6861, 0.7276)


### 6.3. Bổ sung 2 đặc trưng lịch sử

In [ ]:
from pyspark.sql.window import Window

# Lấy trạng thái từng đơn
statuses = df_clean.select(
    "order_id", "delivery_status"
).dropDuplicates(["order_id"])

df_base = (
    df_model
    .join(statuses, "order_id")
    .withColumn("order_day", F.to_date("order_date"))
)

# Chỉ dùng đơn không bị hủy để tính lịch sử
history = df_base.filter(
    F.col("delivery_status") != "Shipping canceled"
)

def add_rate(df, group_col, new_col):
    daily = history.groupBy(group_col, "order_day").agg(
        F.sum("label").alias("late"),
        F.count("*").alias("total")
    )

    # Tạo trục ngày đầy đủ để không bỏ sót ngày
    dates = df.select("order_day").distinct()
    groups = df.select(group_col).distinct()

    daily = groups.crossJoin(dates).join(
        daily, [group_col, "order_day"], "left"
    ).fillna({"late": 0, "total": 0})

    w = Window.partitionBy(group_col).orderBy(
        F.col("order_day").cast("timestamp").cast("long")
    ).rangeBetween(-37*86400, -7*86400)

    daily = daily.withColumn(
        new_col,
        F.try_divide(
          F.sum("late").over(w),
          F.sum("total").over(w)
      )
    )

    return df.join(
        daily.select(group_col, "order_day", new_col),
        [group_col, "order_day"], "left"
    )

df_7 = add_rate(df_base, "shipping_mode", "shipping_mode_late_rate_30d")
df_7 = add_rate(df_7, "order_region", "region_late_rate_30d")

train_rate = df_train.agg(F.avg("label")).first()[0]

df_7 = df_7.fillna({
    "shipping_mode_late_rate_30d": train_rate,
    "region_late_rate_30d": train_rate
})

print("Số đơn:", df_7.count())

Số đơn: 65752


### 6.4. Huấn luyện mô hình 7 đặc trưng

In [ ]:
train_7 = df_7.filter(F.col("order_date") < "2017-01-01")

val_7 = df_7.filter(
    (F.col("order_date") >= "2017-01-01") &
    (F.col("order_date") < "2017-07-01")
)

num_cols_7 = num_cols + [
    "shipping_mode_late_rate_30d",
    "region_late_rate_30d"
]

model_7, result_7 = train_rf(train_7, val_7, num_cols_7)

print("Mô hình 7 đặc trưng:")
print(result_7)

### 6.5. So sánh kết quả

In [ ]:
import pandas as pd

comparison = pd.DataFrame(
    [result_5, result_7],
    columns=["Model", "Accuracy", "Precision",
             "Recall", "Weighted F1", "ROC-AUC"]
)

display(comparison)

,Model,Accuracy,Precision,Recall,Weighted F1,ROC-AUC
0,Random Forest,0.6889,0.7138,0.6889,0.6861,0.7276
1,Random Forest,0.6909,0.7333,0.6909,0.6843,0.7273


### 6.6. Đánh giá mô hình đã chọn trên Test

In [ ]:
pred_5_test = model_5.transform(df_test)

result_5_test = evaluate_model(
    pred_5_test, "RF - 5 đặc trưng (Test)"
)

print(result_5_test)

### 6.7. Kết luận và lựa chọn mô hình


Kết quả thực nghiệm cho thấy mô hình Random Forest với 5 đặc trưng đạt Accuracy 69,49% và ROC-AUC 0,7345 trên tập Test.

Nhóm đã thử mở rộng lên 12 đặc trưng và tối ưu mô hình, tuy nhiên kết quả dự báo không cải thiện đáng kể. Điều này cho thấy việc tăng số lượng đặc trưng không đồng nghĩa với nâng cao độ chính xác.

Vì vậy, nhóm lựa chọn Random Forest với 5 đặc trưng làm mô hình cuối cùng do có kết quả phù hợp hơn và cấu trúc đơn giản, thuận tiện cho triển khai dự báo.

## PHẦN 7. LƯU MÔ HÌNH VÀ KẾT QUẢ DỰ BÁO

### 7.1. Lưu mô hình Random Forest 5 đặc trưng

In [ ]:
model_5.write().overwrite().save(
    base_path + "/models/random_forest_final"
)

print("Đã lưu mô hình thành công!")

Đã lưu mô hình thành công!


### 7.2. Xuất kết quả dự báo trên Test

In [ ]:
pred_5_test.select(
    "order_id", "label", "prediction", "probability"
).write.mode("overwrite").parquet(
    base_path + "/results/prediction_test"
)

print("Đã xuất kết quả dự báo!")

Đã xuất kết quả dự báo!


### 7.3. Kiểm tra kết quả đã lưu

In [ ]:
from pyspark.ml import PipelineModel

# Kiểm tra mô hình
loaded_model = PipelineModel.load(
    base_path + "/models/random_forest_final"
)

# Kiểm tra dữ liệu dự báo
df_result = spark.read.parquet(
    base_path + "/results/prediction_test"
)

print("Số đơn hàng dự báo:", df_result.count())
df_result.show(5, truncate=False)

Số đơn hàng dự báo: 13670
+--------+-----+----------+----------------------------------------+
|order_id|label|prediction|probability                             |
+--------+-----+----------+----------------------------------------+
|62478   |0.0  |0.0       |[0.5696898815315021,0.4303101184684978] |
|62481   |0.0  |0.0       |[0.5696898815315021,0.4303101184684978] |
|62485   |1.0  |0.0       |[0.5823875243440688,0.41761247565593124]|
|62488   |0.0  |0.0       |[0.5823875243440688,0.41761247565593124]|
|62489   |1.0  |0.0       |[0.5675407207774659,0.432459279222534]  |
+--------+-----+----------+----------------------------------------+
only showing top 5 rows


### 7.4. Kết luận

Mô hình Random Forest với 5 đặc trưng đạt Accuracy 69,49% trên tập Test. Mô hình và kết quả dự báo của 13.670 đơn hàng đã được lưu thành công, phục vụ trực quan hóa và đánh giá.

## PHẦN 8. TRỰC QUAN HÓA VÀ PHÂN TÍCH KẾT QUẢ

### 8.1. Chuẩn bị dữ liệu trực quan hóa

In [ ]:
!pip -q install plotly

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df_viz = df_model.toPandas()

df_pred = pred_5_test.select(
    "order_id", "label", "prediction", "probability"
).toPandas()

df_viz["trang_thai"] = df_viz["label"].map({
    0: "Không trễ",
    1: "Giao trễ"
})

print("Dữ liệu phân tích:", df_viz.shape)
print("Dữ liệu dự báo:", df_pred.shape)

Dữ liệu phân tích: (65752, 16)
Dữ liệu dự báo: (13670, 4)


### 8.2. Phân tích tổng quan tình trạng giao hàng

In [ ]:
summary = (
    df_viz["trang_thai"]
    .value_counts()
    .reset_index()
)

summary.columns = ["Trạng thái", "Số đơn"]

fig = px.pie(
    summary,
    names="Trạng thái",
    values="Số đơn",
    hole=0.5,
    title="Tỷ lệ đơn hàng giao trễ và không trễ",
    color="Trạng thái",
    color_discrete_map={
        "Giao trễ": "#EF4444",
        "Không trễ": "#22C55E"
    }
)

fig.update_traces(
    textinfo="percent+label",
    textfont_size=14
)

fig.update_layout(
    template="plotly_white",
    height=450
)

fig.show()

Trong tổng số 65.752 đơn hàng, có 36.048 đơn giao trễ (54,82%) và 29.704 đơn không trễ (45,18%). Tỷ lệ giao trễ chiếm hơn một nửa tổng số đơn hàng, cho thấy tình trạng giao trễ là vấn đề đáng quan tâm trong hoạt động vận chuyển.

### 8.3. Phân tích tình trạng giao trễ theo khu vực

Biểu đồ 2 – Top 10 khu vực có tỷ lệ giao trễ cao nhất

In [ ]:
region_stats = (
    df_viz.groupby("order_region")
    .agg(
        tong_don=("label", "count"),
        so_don_tre=("label", "sum"),
        ty_le_tre=("label", "mean")
    )
    .reset_index()
)

region_stats["ty_le_tre"] *= 100

top10_regions = (
    region_stats[region_stats["tong_don"] >= 100]
    .nlargest(10, "ty_le_tre")
    .sort_values("ty_le_tre")
)

fig = px.bar(
    top10_regions,
    x="ty_le_tre",
    y="order_region",
    orientation="h",
    color="ty_le_tre",
    color_continuous_scale="Reds",
    text="ty_le_tre",
    title="Top 10 khu vực có tỷ lệ giao trễ cao nhất",
    labels={
        "ty_le_tre": "Tỷ lệ giao trễ (%)",
        "order_region": "Khu vực"
    },
    hover_data=["tong_don", "so_don_tre"]
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    coloraxis_showscale=False,
    height=550,
    xaxis_range=[0, 100]
)

fig.show()

Tỷ lệ giao trễ có sự khác biệt giữa các khu vực. Những khu vực có tỷ lệ giao trễ cao cần được ưu tiên theo dõi để cải thiện hiệu quả vận chuyển.

Biểu đồ 3 – Số lượng đơn hàng và đơn giao trễ theo khu vực

In [ ]:
top10_volume = (
    region_stats.nlargest(10, "tong_don")
    .sort_values("tong_don")
)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=top10_volume["order_region"],
    x=top10_volume["tong_don"],
    name="Tổng đơn hàng",
    orientation="h",
    marker_color="#3B82F6"
))

fig.add_trace(go.Bar(
    y=top10_volume["order_region"],
    x=top10_volume["so_don_tre"],
    name="Đơn giao trễ",
    orientation="h",
    marker_color="#EF4444"
))

fig.update_layout(
    title="Top 10 khu vực có nhiều đơn hàng nhất",
    xaxis_title="Số lượng đơn hàng",
    yaxis_title="Khu vực",
    barmode="group",
    template="plotly_white",
    height=550
)

fig.show()

Các khu vực có số lượng đơn hàng lớn thường đóng góp nhiều đơn giao trễ vào tổng số đơn hàng. Vì vậy, cần xem xét đồng thời số lượng và tỷ lệ giao trễ để xác định khu vực cần ưu tiên cải thiện.

### 8.4 Phân tích tình trạng giao trễ theo phương thức vận chuyển

Biểu đồ 4 – Tỷ lệ giao trễ theo phương thức vận chuyển

In [ ]:
mode_stats = (
    df_viz.groupby("shipping_mode")
    .agg(
        tong_don=("label", "count"),
        so_don_tre=("label", "sum"),
        ty_le_tre=("label", "mean")
    )
    .reset_index()
)

mode_stats["ty_le_tre"] *= 100
mode_stats = mode_stats.sort_values("ty_le_tre", ascending=False)

fig = px.bar(
    mode_stats,
    x="shipping_mode",
    y="ty_le_tre",
    color="shipping_mode",
    text="ty_le_tre",
    title="Tỷ lệ giao trễ theo phương thức vận chuyển",
    labels={
        "shipping_mode": "Phương thức vận chuyển",
        "ty_le_tre": "Tỷ lệ giao trễ (%)"
    },
    hover_data=["tong_don", "so_don_tre"]
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    showlegend=False,
    yaxis_range=[0, 105],
    height=450
)

fig.show()

First Class có tỷ lệ giao trễ cao nhất (95,27%), tiếp theo là Second Class (76,72%), Same Day (46,15%) và Standard Class (38,13%). Điều này cho thấy phương thức vận chuyển có mối liên hệ rõ rệt với nguy cơ giao hàng trễ.

Biểu đồ 5 – Số đơn giao trễ và không trễ theo phương thức vận chuyển

In [ ]:
mode_count = (
    df_viz.groupby(["shipping_mode", "trang_thai"])
    .size()
    .reset_index(name="so_don")
)

fig = px.bar(
    mode_count,
    x="shipping_mode",
    y="so_don",
    color="trang_thai",
    barmode="group",
    text="so_don",
    title="Số đơn giao trễ và không trễ theo phương thức vận chuyển",
    labels={
        "shipping_mode": "Phương thức vận chuyển",
        "so_don": "Số đơn hàng",
        "trang_thai": "Trạng thái giao hàng"
    },
    color_discrete_map={
        "Giao trễ": "#EF4444",
        "Không trễ": "#22C55E"
    }
)

fig.update_traces(textposition="outside")

fig.update_layout(
    template="plotly_white",
    height=500,
    xaxis_title="Phương thức vận chuyển",
    yaxis_title="Số lượng đơn hàng"
)

fig.show()

Standard Class có số lượng đơn hàng lớn nhất và cũng ghi nhận nhiều đơn giao trễ nhất (14.995 đơn). Tuy nhiên, First Class có tỷ lệ giao trễ cao nhất. Vì vậy, cần kết hợp cả số lượng và tỷ lệ giao trễ khi đánh giá các phương thức vận chuyển.

### 8.5. Phân tích tình trạng giao trễ theo thời gian

Biểu đồ 6 – Xu hướng tỷ lệ giao trễ theo tháng

In [ ]:
monthly_stats = (
    df_viz.groupby(["order_year", "order_month"])
    .agg(
        tong_don=("label", "count"),
        ty_le_tre=("label", "mean")
    )
    .reset_index()
)

monthly_stats["ty_le_tre"] *= 100

monthly_stats["thoi_gian"] = pd.to_datetime(
    monthly_stats[["order_year", "order_month"]]
    .rename(columns={
        "order_year": "year",
        "order_month": "month"
    })
    .assign(day=1)
)

monthly_stats = monthly_stats.sort_values("thoi_gian")

fig = px.line(
    monthly_stats,
    x="thoi_gian",
    y="ty_le_tre",
    markers=True,
    title="Xu hướng tỷ lệ giao trễ theo tháng",
    labels={
        "thoi_gian": "Thời gian",
        "ty_le_tre": "Tỷ lệ giao trễ (%)"
    },
    hover_data=["tong_don"]
)

fig.update_traces(
    line_color="#2563EB",
    line_width=3,
    marker_size=7
)

fig.update_layout(
    template="plotly_white",
    height=500,
    yaxis_range=[0, 100]
)

fig.show()

Biểu đồ thể hiện sự biến động của tỷ lệ giao trễ qua các tháng. Việc theo dõi xu hướng này giúp xác định những giai đoạn có nguy cơ giao trễ cao để hỗ trợ lập kế hoạch vận chuyển.

Biểu đồ 7 – Tỷ lệ giao trễ theo ngày trong tuần

In [ ]:
weekday_stats = (
    df_viz.groupby("order_day_of_week")
    .agg(
        tong_don=("label", "count"),
        ty_le_tre=("label", "mean")
    )
    .reset_index()
)

weekday_stats["ty_le_tre"] *= 100
weekday_stats = weekday_stats.sort_values("order_day_of_week")

fig = px.bar(
    weekday_stats,
    x="order_day_of_week",
    y="ty_le_tre",
    color="ty_le_tre",
    color_continuous_scale="Blues",
    text="ty_le_tre",
    title="Tỷ lệ giao trễ theo ngày trong tuần",
    labels={
        "order_day_of_week": "Ngày trong tuần (theo mã dữ liệu)",
        "ty_le_tre": "Tỷ lệ giao trễ (%)"
    },
    hover_data=["tong_don"]
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    coloraxis_showscale=False,
    height=450,
    yaxis_range=[0, 100]
)

fig.show()

Tỷ lệ giao trễ được thống kê theo từng ngày trong tuần, giúp đánh giá liệu thời điểm đặt hàng có liên quan đến nguy cơ giao trễ hay không.

### 8.6. Phân tích tình trạng giao trễ theo đặc điểm đơn hàng

Biểu đồ 8 – Tỷ lệ giao trễ theo giá trị đơn hàng

In [ ]:
# Chia giá trị đơn hàng thành 5 nhóm theo phân vị
df_viz["nhom_gia_tri"] = pd.qcut(
    df_viz["total_order_value"],
    q=5,
    duplicates="drop"
)

value_stats = (
    df_viz.groupby("nhom_gia_tri", observed=True)
    .agg(
        tong_don=("label", "count"),
        ty_le_tre=("label", "mean")
    )
    .reset_index()
)

value_stats["ty_le_tre"] *= 100
value_stats["nhom_gia_tri"] = value_stats["nhom_gia_tri"].astype(str)

fig = px.bar(
    value_stats,
    x="nhom_gia_tri",
    y="ty_le_tre",
    color="ty_le_tre",
    color_continuous_scale="Blues",
    text="ty_le_tre",
    title="Tỷ lệ giao trễ theo nhóm giá trị đơn hàng",
    labels={
        "nhom_gia_tri": "Nhóm giá trị đơn hàng",
        "ty_le_tre": "Tỷ lệ giao trễ (%)"
    },
    hover_data=["tong_don"]
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    coloraxis_showscale=False,
    height=450,
    yaxis_range=[0, 100],
    xaxis_tickangle=-20
)

fig.show()

Biểu đồ so sánh tỷ lệ giao trễ giữa các nhóm giá trị đơn hàng, qua đó đánh giá liệu giá trị đơn hàng có liên quan đến tình trạng giao trễ hay không.

Biểu đồ 9 – Tỷ lệ giao trễ theo số lượng sản phẩm

In [ ]:
# Chia đơn hàng theo tổng số lượng sản phẩm
df_viz["nhom_so_luong"] = pd.cut(
    df_viz["total_quantity"],
    bins=[0, 1, 2, 5, 10, float("inf")],
    labels=["1", "2", "3–5", "6–10", "Trên 10"]
)

quantity_stats = (
    df_viz.groupby("nhom_so_luong", observed=True)
    .agg(
        tong_don=("label", "count"),
        ty_le_tre=("label", "mean")
    )
    .reset_index()
)

quantity_stats["ty_le_tre"] *= 100

fig = px.bar(
    quantity_stats,
    x="nhom_so_luong",
    y="ty_le_tre",
    color="ty_le_tre",
    color_continuous_scale="Oranges",
    text="ty_le_tre",
    title="Tỷ lệ giao trễ theo số lượng sản phẩm",
    labels={
        "nhom_so_luong": "Số lượng sản phẩm trong đơn hàng",
        "ty_le_tre": "Tỷ lệ giao trễ (%)"
    },
    hover_data=["tong_don"]
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    coloraxis_showscale=False,
    height=450,
    yaxis_range=[0, 100]
)

fig.show()

Biểu đồ thể hiện tỷ lệ giao trễ theo các nhóm số lượng sản phẩm, giúp xem xét liệu những đơn hàng có nhiều sản phẩm có xu hướng giao trễ cao hơn hay không.

### 8.7. Trực quan hóa và đánh giá mô hình dự báo

Biểu đồ 10 – Ma trận nhầm lẫn (Confusion Matrix)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    df_pred["label"].astype(int),
    df_pred["prediction"].astype(int),
    labels=[0, 1]
)

fig = px.imshow(
    cm,
    x=["Không trễ", "Giao trễ"],
    y=["Không trễ", "Giao trễ"],
    text_auto=True,
    color_continuous_scale="Blues",
    title="Ma trận nhầm lẫn - Random Forest"
)

fig.update_layout(
    xaxis_title="Kết quả dự đoán",
    yaxis_title="Kết quả thực tế",
    template="plotly_white",
    height=450
)

fig.show()

Ma trận nhầm lẫn cho thấy số lượng đơn hàng được dự đoán chính xác và các trường hợp dự đoán sai. Đặc biệt, các đơn thực tế giao trễ nhưng được dự đoán không trễ là những trường hợp cần quan tâm vì mô hình chưa phát hiện được nguy cơ.

Biểu đồ 11 – Đường cong ROC-AUC

In [ ]:
from sklearn.metrics import roc_curve, auc

# Lấy xác suất dự báo giao trễ (lớp 1)
y_true = df_pred["label"].astype(int)
y_score = df_pred["probability"].apply(lambda p: float(p[1]))

fpr, tpr, thresholds = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fpr,
    y=tpr,
    mode="lines",
    name=f"Random Forest (AUC = {roc_auc:.4f})",
    line=dict(color="#2563EB", width=3)
))

fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    name="Dự đoán ngẫu nhiên",
    line=dict(color="gray", dash="dash")
))

fig.update_layout(
    title="Đường cong ROC - Random Forest",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    height=500
)

fig.show()

print(f"ROC-AUC: {roc_auc:.4f}")

ROC-AUC: 0.7345


Mô hình Random Forest với 5 đặc trưng đạt ROC-AUC khoảng 0,7345 trên tập Test, cho thấy khả năng phân biệt giữa hai nhóm đơn hàng tốt hơn dự đoán ngẫu nhiên, nhưng vẫn còn khả năng cải thiện.

Biểu đồ 12 – So sánh hiệu quả mô hình Random Forest

In [ ]:
comparison = pd.DataFrame({
    "Mô hình": ["RF - 12 đặc trưng", "RF - 5 đặc trưng"],
    "Accuracy": [0.6948, 0.6949],
    "Precision": [0.7210, 0.7210],
    "Recall": [0.6948, 0.6949],
    "Weighted F1": [0.6929, 0.6930],
    "ROC-AUC": [0.7309, 0.7345]
})

comparison_long = comparison.melt(
    id_vars="Mô hình",
    var_name="Chỉ số",
    value_name="Giá trị"
)

fig = px.bar(
    comparison_long,
    x="Chỉ số",
    y="Giá trị",
    color="Mô hình",
    barmode="group",
    text="Giá trị",
    title="So sánh hiệu quả Random Forest trên tập Test",
    color_discrete_sequence=["#94A3B8", "#2563EB"]
)

fig.update_traces(
    texttemplate="%{text:.4f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Giá trị chỉ số",
    yaxis_range=[0, 0.85],
    height=500
)

fig.show()

Mô hình Random Forest với 5 đặc trưng đạt Accuracy 69,49%, Weighted F1 0,6930 và ROC-AUC 0,7345 trên tập Test. Kết quả tương đương mô hình 12 đặc trưng, trong khi số lượng đặc trưng được giảm đáng kể. Vì vậy, nhóm lựa chọn mô hình 5 đặc trưng để sử dụng cuối cùng.

### 8.8. Nhận xét và kết luận

Thông qua 12 biểu đồ trực quan hóa, nhóm đã phân tích tình trạng giao trễ theo khu vực, phương thức vận chuyển, thời gian và đặc điểm đơn hàng. Kết quả cho thấy tỷ lệ giao trễ chiếm 54,82% tổng số đơn hàng và có sự khác biệt giữa các nhóm dữ liệu.

Mô hình Random Forest với 5 đặc trưng đạt Accuracy 69,49% và ROC-AUC 0,7345 trên tập Test. Kết quả cho thấy mô hình có khả năng hỗ trợ dự báo nguy cơ giao trễ, tuy nhiên độ chính xác vẫn cần được cải thiện.

## PHẦN 9. XÂY DỰNG DASHBOARD TƯƠNG TÁC BẰNG STREAMLIT

### 9.1. Xuất dữ liệu phục vụ Dashboard

In [ ]:
import os

dashboard_path = base_path + "/dashboard"
os.makedirs(dashboard_path, exist_ok=True)

# Xuất dữ liệu phân tích
df_viz.to_csv(
    dashboard_path + "/data_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

# Xuất kết quả dự báo
df_pred["probability_late"] = df_pred["probability"].apply(
    lambda p: float(p[1])
)

df_pred[
    ["order_id", "label", "prediction", "probability_late"]
].to_csv(
    dashboard_path + "/prediction_test.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất dữ liệu Dashboard!")
print("Thư mục:", dashboard_path)

Đã xuất dữ liệu Dashboard!
Thư mục: /content/mydrive_new/MyDrive/BigData_Delivery_Prediction/dashboard


### 9.2. Cài đặt Streamlit

In [1]:
!pip -q install streamlit plotly

### 9.3. Xây dựng ứng dụng Dashboard bằng Streamlit

In [4]:
import os

# Khôi phục đường dẫn khi mở lại Google Colab
if "dashboard_path" not in globals():
    if "base_path" not in globals():
        raise RuntimeError(
            "Chưa có base_path. Hãy chạy ô kết nối Drive và khai báo base_path trước ô 9.3."
        )
    dashboard_path = os.path.join(base_path, "dashboard")
os.makedirs(dashboard_path, exist_ok=True)

app_code = r'''
import streamlit as st
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve, auc

st.set_page_config(
    page_title="Delivery Intelligence",
    page_icon="📦",
    layout="wide",
    initial_sidebar_state="expanded"
)

NAVY = "#14263F"
BLUE = "#2563EB"
RED = "#EF4444"
GREEN = "#10B981"
MUTED = "#64748B"

# ==================== GIAO DIỆN ====================
st.markdown("""
<style>
.stApp {
    background: #F5F7FB;
}
.block-container {
    max-width: 1450px;
    padding-top: 1.4rem;
    padding-bottom: 2rem;
}
[data-testid="stSidebar"] {
    background: #14263F;
}
[data-testid="stSidebar"] * {
    color: #EAF0FA !important;
}
[data-testid="stSidebar"] [data-baseweb="select"] * {
    color: #172B4D !important;
}
[data-testid="stSidebar"] [data-baseweb="tag"] {
    background: #2563EB !important;
}
[data-testid="stSidebar"] [data-baseweb="tag"] * {
    color: white !important;
}
[data-testid="stSidebar"] [data-testid="stRadio"] label {
    padding: 7px 0;
}
h1, h2, h3 {
    color: #172B4D !important;
}
.hero {
    background: linear-gradient(110deg,#14263F,#244B85);
    padding: 27px 30px;
    border-radius: 18px;
    margin-bottom: 24px;
}
.hero .eyebrow {
    color: #A9C8FF;
    font-size: 12px;
    letter-spacing: 2px;
    font-weight: 700;
}
.hero h1 {
    color: white !important;
    font-size: 29px;
    margin: 9px 0;
}
.hero p {
    color: #DCE8F8;
    font-size: 14px;
    margin: 0;
}
.kpi {
    background: white;
    border: 1px solid #E7ECF4;
    border-radius: 15px;
    padding: 20px;
    box-shadow: 0 4px 15px rgba(15,23,42,.035);
    min-height: 130px;
}
.kpi-label {
    color: #64748B;
    font-size: 13px;
    font-weight: 600;
}
.kpi-value {
    font-size: 30px;
    font-weight: 800;
    color: #14263F;
    margin-top: 12px;
}
.kpi-note {
    color: #94A3B8;
    font-size: 11px;
    margin-top: 5px;
}
.section-heading {
    font-size: 19px;
    font-weight: 750;
    color: #172B4D;
    margin: 22px 0 12px;
}
.insight {
    background: #EFF6FF;
    border-left: 4px solid #2563EB;
    border-radius: 9px;
    padding: 17px 19px;
    color: #1E3A5F;
    line-height: 1.7;
}
[data-testid="stPlotlyChart"] {
    background: white;
    border: 1px solid #E7ECF4;
    border-radius: 15px;
    padding: 10px;
}
footer {
    visibility: hidden;
}
</style>
""", unsafe_allow_html=True)

# ==================== ĐỌC DỮ LIỆU ====================
@st.cache_data
def load_data():
    data = pd.read_csv("data_analysis.csv")
    prediction = pd.read_csv("prediction_test.csv")
    return data, prediction

df, pred = load_data()

df["trang_thai"] = df["label"].map({
    0: "Không trễ",
    1: "Giao trễ"
})

# ==================== HÀM GIAO DIỆN ====================
def kpi(title, value, note="", color=NAVY):
    st.markdown(
        f"""
        <div class="kpi">
            <div class="kpi-label">{title}</div>
            <div class="kpi-value" style="color:{color}">{value}</div>
            <div class="kpi-note">{note}</div>
        </div>
        """,
        unsafe_allow_html=True
    )

def chart(fig, height=365):
    fig.update_layout(
        template="plotly_white",
        height=height,
        margin=dict(l=30, r=25, t=65, b=45),
        font=dict(family="Arial", size=12, color="#475569"),
        paper_bgcolor="white",
        plot_bgcolor="white",
        title_font=dict(size=15, color=NAVY),
        legend=dict(orientation="h", y=-0.2),
        colorway=[BLUE, RED, GREEN, "#8B5CF6"],
    )
    st.plotly_chart(fig, use_container_width=True)

def insight(message):
    st.markdown(
        f'<div class="insight">{message}</div>',
        unsafe_allow_html=True
    )

def stats_by(column):
    result = data.groupby(column).agg(
        tong_don=("label", "count"),
        don_tre=("label", "sum"),
        ty_le=("label", "mean")
    ).reset_index()
    result["ty_le"] *= 100
    return result

# ==================== MENU ====================
st.sidebar.markdown("## 📦 DELIVERY")
st.sidebar.markdown("### INTELLIGENCE")
st.sidebar.caption("DATA ANALYTICS PLATFORM")
st.sidebar.divider()

pages = [
    "Tổng quan",
    "Khu vực",
    "Vận chuyển",
    "Thời gian",
    "Đơn hàng",
    "Dự báo ML",
    "Dự báo đơn hàng mới"
]

page = st.sidebar.radio(
    "DANH MỤC",
    pages,
    label_visibility="collapsed"
)

st.sidebar.divider()

# Bộ lọc thu gọn, không hiển thị hàng loạt thẻ
with st.sidebar.expander("⚙️ Bộ lọc dữ liệu", expanded=False):
    years = sorted(df["order_year"].dropna().unique().tolist())
    selected_years = st.multiselect(
        "Năm",
        years,
        default=years
    )

    regions = sorted(df["order_region"].dropna().unique().tolist())
    selected_region = st.selectbox(
        "Khu vực",
        ["Tất cả"] + regions
    )

    modes = sorted(df["shipping_mode"].dropna().unique().tolist())
    selected_mode = st.selectbox(
        "Vận chuyển",
        ["Tất cả"] + modes
    )

data = df[df["order_year"].isin(selected_years)].copy()

if selected_region != "Tất cả":
    data = data[data["order_region"] == selected_region]

if selected_mode != "Tất cả":
    data = data[data["shipping_mode"] == selected_mode]

st.sidebar.caption("DataCo Smart Supply Chain")
st.sidebar.caption("GROUP 6 • BIG DATA")

# ==================== HEADER ====================
st.markdown("""
<div class="hero">
    <div class="eyebrow">SUPPLY CHAIN ANALYTICS</div>
    <h1>Phân tích & Dự báo trễ hạn giao hàng</h1>
    <p>Dashboard phân tích dữ liệu DataCo Smart Supply Chain | PySpark • Random Forest • Plotly</p>
</div>
""", unsafe_allow_html=True)

if data.empty and page not in ["Dự báo ML", "Dự báo đơn hàng mới"]:
    st.warning("Không có đơn hàng phù hợp với bộ lọc hiện tại.")
    st.stop()

# ==================== TỔNG QUAN ====================
if page == "Tổng quan":
    total = len(data)
    late = int(data["label"].sum())
    ontime = total - late
    rate = late / total * 100

    c1, c2, c3, c4 = st.columns(4)

    with c1:
        kpi("TỔNG ĐƠN HÀNG", f"{total:,}", "Số đơn trong phạm vi phân tích")
    with c2:
        kpi("ĐƠN GIAO TRỄ", f"{late:,}", "Đơn có nguy cơ trễ", RED)
    with c3:
        kpi("ĐƠN KHÔNG TRỄ", f"{ontime:,}", "Đơn giao đúng hạn", GREEN)
    with c4:
        kpi("TỶ LỆ GIAO TRỄ", f"{rate:.2f}%", "Tỷ lệ trên tổng đơn", BLUE)

    st.markdown(
        '<div class="section-heading">Tình trạng giao hàng</div>',
        unsafe_allow_html=True
    )

    left, right = st.columns([1.05, 1], gap="medium")

    with left:
        fig = go.Figure(go.Pie(
            labels=["Giao trễ", "Không trễ"],
            values=[late, ontime],
            hole=0.76,
            marker_colors=[RED, GREEN],
            textinfo="percent",
            textposition="inside",
            sort=False
        ))
        fig.update_layout(
            title="Cơ cấu trạng thái đơn hàng",
            annotations=[dict(
                text=f"<b>{rate:.2f}%</b><br>Giao trễ",
                x=0.5, y=0.5,
                showarrow=False,
                font=dict(size=20, color=NAVY)
            )]
        )
        chart(fig, 350)

    with right:
        st.markdown("#### Tổng hợp phân tích")
        st.write(
            f"Trong **{total:,} đơn hàng**, có **{late:,} đơn giao trễ**, "
            f"tương ứng **{rate:.2f}%**."
        )
        st.write(
            f"Dữ liệu đang bao gồm **{data['order_region'].nunique()} khu vực** "
            f"và **{data['shipping_mode'].nunique()} phương thức vận chuyển**."
        )

        if rate > 50:
            insight(
                "Tỷ lệ giao trễ đang vượt 50%. "
                "Cần ưu tiên phân tích phương thức vận chuyển và "
                "khu vực có tỷ lệ giao trễ cao."
            )
        else:
            insight(
                "Tỷ lệ giao trễ trong phạm vi đang chọn không vượt 50%. "
                "Có thể tiếp tục phân tích các nhóm đơn hàng có rủi ro cao."
            )

# ==================== KHU VỰC ====================
elif page == "Khu vực":
    st.subheader("Phân tích giao hàng theo khu vực")

    stats = stats_by("order_region")
    top_rate = (
        stats[stats["tong_don"] >= 100]
        .nlargest(10, "ty_le")
        .sort_values("ty_le")
    )

    if not top_rate.empty:
        fig = px.bar(
            top_rate,
            x="ty_le",
            y="order_region",
            orientation="h",
            text="ty_le",
            title="Top 10 khu vực có tỷ lệ giao trễ cao nhất",
            labels={"ty_le": "Tỷ lệ (%)", "order_region": "Khu vực"}
        )
        fig.update_traces(
            marker_color=BLUE,
            texttemplate="%{text:.1f}%"
        )
        chart(fig, 450)
    else:
        st.info("Không đủ khu vực có từ 100 đơn hàng.")

    top_volume = stats.nlargest(10, "tong_don")

    fig = go.Figure()
    fig.add_bar(
        x=top_volume["order_region"],
        y=top_volume["tong_don"],
        name="Tổng đơn",
        marker_color=BLUE
    )
    fig.add_bar(
        x=top_volume["order_region"],
        y=top_volume["don_tre"],
        name="Đơn trễ",
        marker_color=RED
    )
    fig.update_layout(
        title="Số đơn hàng và đơn giao trễ theo khu vực",
        barmode="group"
    )
    chart(fig, 420)

    # Phân tích đa chiều: khu vực x phương thức vận chuyển
    st.markdown("#### Bản đồ nhiệt tỷ lệ giao trễ theo khu vực và vận chuyển")
    min_orders_heat = st.slider(
        "Số đơn tối thiểu cho mỗi tổ hợp", 10, 300, 50, 10,
        key="heat_min_orders"
    )
    heat_stats = data.groupby(["order_region", "shipping_mode"]).agg(
        tong_don=("label", "count"), ty_le=("label", "mean")
    ).reset_index()
    heat_stats = heat_stats[heat_stats["tong_don"] >= min_orders_heat].copy()
    if not heat_stats.empty:
        heat_stats["ty_le"] *= 100
        heat_rate = heat_stats.pivot(index="order_region", columns="shipping_mode", values="ty_le")
        heat_count = heat_stats.pivot(index="order_region", columns="shipping_mode", values="tong_don")
        heat_rate = heat_rate.sort_index()
        heat_count = heat_count.reindex(index=heat_rate.index, columns=heat_rate.columns)
        fig = go.Figure(go.Heatmap(
            z=heat_rate.to_numpy(), x=heat_rate.columns.tolist(), y=heat_rate.index.tolist(),
            customdata=heat_count.to_numpy(), colorscale="YlOrRd",
            colorbar=dict(title="Tỷ lệ (%)"),
            hovertemplate="Khu vực: %{y}<br>Vận chuyển: %{x}<br>Tỷ lệ trễ: %{z:.2f}%<br>Số đơn: %{customdata:.0f}<extra></extra>"
        ))
        fig.update_layout(title="Tỷ lệ giao trễ theo từng tổ hợp", yaxis=dict(autorange="reversed"))
        chart(fig, max(450, 26 * len(heat_rate) + 130))
    else:
        st.info("Không có tổ hợp đủ số đơn theo ngưỡng đã chọn.")

    # Phân tích sâu theo quốc gia, tránh kết luận từ nhóm có quá ít đơn
    if "order_country" in data.columns:
        st.markdown("#### Phân tích giao trễ theo quốc gia")
        c_top, c_min = st.columns(2)
        with c_top:
            top_n_countries = st.slider("Số quốc gia hiển thị", 5, 20, 10, key="country_top_n")
        with c_min:
            min_orders_country = st.number_input(
                "Số đơn tối thiểu mỗi quốc gia", min_value=1, max_value=10000,
                value=100, step=50, key="country_min_orders"
            )
        country_stats = data.dropna(subset=["order_country"]).groupby("order_country").agg(
            tong_don=("label", "count"), don_tre=("label", "sum"), ty_le=("label", "mean")
        ).reset_index()
        country_stats = country_stats[country_stats["tong_don"] >= min_orders_country].copy()
        if not country_stats.empty:
            country_stats["ty_le"] *= 100
            top_countries = country_stats.nlargest(top_n_countries, "ty_le").sort_values("ty_le")
            fig = px.bar(
                top_countries, x="ty_le", y="order_country", orientation="h",
                color="tong_don", color_continuous_scale="Blues",
                custom_data=["tong_don", "don_tre"],
                labels={"ty_le": "Tỷ lệ giao trễ (%)", "order_country": "Quốc gia", "tong_don": "Tổng đơn"},
                title="Các quốc gia có tỷ lệ giao trễ cao (đủ ngưỡng số đơn)"
            )
            fig.update_traces(hovertemplate="Quốc gia: %{y}<br>Tỷ lệ trễ: %{x:.2f}%<br>Tổng đơn: %{customdata[0]}<br>Đơn trễ: %{customdata[1]}<extra></extra>")
            chart(fig, max(400, top_n_countries * 31 + 130))
        else:
            st.info("Không có quốc gia đủ số đơn theo ngưỡng đã chọn.")

    highest = stats.loc[stats["ty_le"].idxmax()]
    insight(
        f"Khu vực <b>{highest['order_region']}</b> có tỷ lệ giao trễ "
        f"cao nhất trong phạm vi lọc: <b>{highest['ty_le']:.2f}%</b>."
    )

# ==================== VẬN CHUYỂN ====================
elif page == "Vận chuyển":
    st.subheader("Hiệu quả phương thức vận chuyển")

    stats = stats_by("shipping_mode")
    c1, c2 = st.columns(2, gap="medium")

    with c1:
        fig = px.bar(
            stats.sort_values("ty_le", ascending=False),
            x="shipping_mode",
            y="ty_le",
            text="ty_le",
            title="Tỷ lệ giao trễ theo phương thức"
        )
        fig.update_traces(
            marker_color=BLUE,
            texttemplate="%{text:.2f}%"
        )
        chart(fig, 400)

    with c2:
        counts = data.groupby(
            ["shipping_mode", "trang_thai"]
        ).size().reset_index(name="so_don")

        fig = px.bar(
            counts,
            x="shipping_mode",
            y="so_don",
            color="trang_thai",
            barmode="group",
            title="So sánh số đơn trễ và không trễ",
            color_discrete_map={
                "Giao trễ": RED,
                "Không trễ": GREEN
            }
        )
        chart(fig, 400)

    highest = stats.loc[stats["ty_le"].idxmax()]
    insight(
        f"Phương thức <b>{highest['shipping_mode']}</b> có tỷ lệ giao trễ "
        f"cao nhất: <b>{highest['ty_le']:.2f}%</b>."
    )

# ==================== THỜI GIAN ====================
elif page == "Thời gian":
    st.subheader("Xu hướng giao hàng theo thời gian")

    monthly = data.groupby(
        ["order_year", "order_month"]
    ).agg(
        ty_le=("label", "mean")
    ).reset_index()

    monthly["ty_le"] *= 100
    monthly["date"] = pd.to_datetime(
        dict(
            year=monthly["order_year"],
            month=monthly["order_month"],
            day=1
        )
    )

    fig = px.line(
        monthly.sort_values("date"),
        x="date",
        y="ty_le",
        markers=True,
        title="Xu hướng tỷ lệ giao trễ theo tháng",
        labels={"date": "Thời gian", "ty_le": "Tỷ lệ (%)"}
    )
    fig.update_traces(line_color=BLUE, line_width=3)
    chart(fig, 400)

    weekday = stats_by("order_day_of_week")

    fig = px.bar(
        weekday.sort_values("order_day_of_week"),
        x="order_day_of_week",
        y="ty_le",
        text="ty_le",
        title="Tỷ lệ giao trễ theo ngày trong tuần"
    )
    fig.update_traces(
        marker_color=BLUE,
        texttemplate="%{text:.1f}%"
    )
    chart(fig, 370)

    # Xu hướng theo thời gian kết hợp phương thức vận chuyển
    st.markdown("#### So sánh xu hướng giao trễ theo phương thức vận chuyển")
    monthly_mode = data.groupby(["order_year", "order_month", "shipping_mode"]).agg(
        tong_don=("label", "count"), ty_le=("label", "mean")
    ).reset_index()
    monthly_mode["ty_le"] *= 100
    monthly_mode["date"] = pd.to_datetime(dict(
        year=monthly_mode["order_year"], month=monthly_mode["order_month"], day=1
    ))
    mode_options = sorted(monthly_mode["shipping_mode"].dropna().unique().tolist())
    chosen_modes = st.multiselect(
        "Chọn phương thức để so sánh", mode_options, default=mode_options,
        key="trend_shipping_modes"
    )
    monthly_mode = monthly_mode[monthly_mode["shipping_mode"].isin(chosen_modes)]
    if not monthly_mode.empty:
        fig = px.line(
            monthly_mode.sort_values("date"), x="date", y="ty_le", color="shipping_mode",
            markers=True, custom_data=["tong_don"],
            labels={"date": "Tháng đặt hàng", "ty_le": "Tỷ lệ giao trễ (%)", "shipping_mode": "Vận chuyển"},
            title="Xu hướng giao trễ theo tháng của từng phương thức"
        )
        fig.update_traces(hovertemplate="Tháng: %{x|%m/%Y}<br>Tỷ lệ trễ: %{y:.2f}%<br>Số đơn: %{customdata[0]}<extra></extra>")
        chart(fig, 430)
    else:
        st.info("Hãy chọn ít nhất một phương thức vận chuyển.")

    insight(
        "Biểu đồ giúp theo dõi sự thay đổi tỷ lệ giao trễ "
        "theo tháng và theo ngày trong tuần."
    )

# ==================== ĐƠN HÀNG ====================
elif page == "Đơn hàng":
    st.subheader("Phân tích đặc điểm đơn hàng")

    temp = data.copy()

    temp["nhom_gia_tri"] = pd.qcut(
        temp["total_order_value"],
        q=5,
        duplicates="drop"
    )

    value_stats = temp.groupby(
        "nhom_gia_tri", observed=True
    )["label"].mean().reset_index()

    value_stats["label"] *= 100
    value_stats["nhom_gia_tri"] = (
        value_stats["nhom_gia_tri"].astype(str)
    )

    fig = px.bar(
        value_stats,
        x="nhom_gia_tri",
        y="label",
        text="label",
        title="Tỷ lệ giao trễ theo nhóm giá trị đơn hàng",
        labels={"label": "Tỷ lệ (%)", "nhom_gia_tri": "Nhóm giá trị"}
    )
    fig.update_traces(
        marker_color=BLUE,
        texttemplate="%{text:.1f}%"
    )
    chart(fig, 390)

    temp["nhom_so_luong"] = pd.cut(
        temp["total_quantity"],
        bins=[0, 1, 2, 5, 10, np.inf],
        labels=["1", "2", "3–5", "6–10", "Trên 10"]
    )

    quantity_stats = temp.groupby(
        "nhom_so_luong", observed=True
    )["label"].mean().reset_index()

    quantity_stats["label"] *= 100

    fig = px.bar(
        quantity_stats,
        x="nhom_so_luong",
        y="label",
        text="label",
        title="Tỷ lệ giao trễ theo số lượng sản phẩm",
        labels={"label": "Tỷ lệ (%)", "nhom_so_luong": "Số lượng"}
    )
    fig.update_traces(
        marker_color=BLUE,
        texttemplate="%{text:.1f}%"
    )
    chart(fig, 370)

    insight(
        "Phân tích giúp xem xét mối liên hệ giữa giá trị, "
        "số lượng sản phẩm và tỷ lệ giao trễ."
    )


# ==================== DỰ BÁO ĐƠN HÀNG MỚI ====================
elif page == "Dự báo đơn hàng mới":
    st.subheader("Dự báo nguy cơ giao trễ cho đơn hàng mới")
    st.caption("Dự báo bằng Pipeline Random Forest 5 đặc trưng đã huấn luyện trong Google Colab.")

    model_dir = Path(__file__).resolve().parent.parent / "models" / "random_forest_final"
    if not model_dir.is_dir():
        st.error("Chưa tìm thấy mô hình đã lưu tại: " + str(model_dir))
        st.info("Hãy chạy mục 7.1 trong notebook để lưu Pipeline, sau đó chạy lại Dashboard.")
        st.stop()

    @st.cache_resource(show_spinner="Đang nạp mô hình Spark...")
    def load_spark_pipeline(model_path):
        from pyspark.sql import SparkSession
        from pyspark.ml import PipelineModel
        spark_session = (SparkSession.builder
                         .appName("DataCo-New-Order-Prediction")
                         .master("local[*]")
                         .getOrCreate())
        return spark_session, PipelineModel.load(model_path)

    # Danh sách danh mục từ dữ liệu đã xuất; giữ nguyên giá trị gốc để dự báo.
    modes_new = sorted(df["shipping_mode"].dropna().astype(str).unique().tolist())
    regions_new = sorted(df["order_region"].dropna().astype(str).unique().tolist())
    countries_new = (
        sorted(df["order_country"].dropna().astype(str).unique().tolist())
        if "order_country" in df.columns else []
    )

    if not countries_new:
        st.error("data_analysis.csv chưa có cột order_country. Cần xuất thêm cột này trước khi dự báo.")
        st.stop()

    # Chỉ thay đổi nhãn hiển thị; giá trị truyền vào Pipeline vẫn là dữ liệu gốc.
    COUNTRY_DISPLAY = {
    "Afganist�n": "Afganistán",
    "Arabia Saud�": "Arabia Saudí",
    "Azerbaiy�n": "Azerbaiyán",
    "Banglad�s": "Bangladés",
    "Bar�in": "Baréin",
    "Ben�n": "Benín",
    "But�n": "Bután",
    "B�lgica": "Bélgica",
    "Camer�n": "Camerún",
    "Emiratos �rabes Unidos": "Emiratos Árabes Unidos",
    "Espa�a": "España",
    "Etiop�a": "Etiopía",
    "Gab�n": "Gabón",
    "Hait�": "Haití",
    "Hungr�a": "Hungría",
    "Ir�n": "Irán",
    "Jap�n": "Japón",
    "Kazajist�n": "Kazajistán",
    "Kirguist�n": "Kirguistán",
    "L�bano": "Líbano",
    "M�xico": "México",
    "N�ger": "Níger",
    "Om�n": "Omán",
    "Pakist�n": "Pakistán",
    "Panam�": "Panamá",
    "Pap�a Nueva Guinea": "Papúa Nueva Guinea",
    "Pa�ses Bajos": "Países Bajos",
    "Per�": "Perú",
    "Rep�blica Centroafricana": "República Centroafricana",
    "Rep�blica Checa": "República Checa",
    "Rep�blica Democr�tica del Congo": "República Democrática del Congo",
    "Rep�blica Dominicana": "República Dominicana",
    "Rep�blica de Gambia": "República de Gambia",
    "Rep�blica del Congo": "República del Congo",
    "Sud�n": "Sudán",
    "Sud�n del Sur": "Sudán del Sur",
    "S�hara Occidental": "Sáhara Occidental",
    "Taiw�n": "Taiwán",
    "Tayikist�n": "Tayikistán",
    "Turkmenist�n": "Turkmenistán",
    "Turqu�a": "Turquía",
    "T�nez": "Túnez",
    "Uzbekist�n": "Uzbekistán",
}

    # Ràng buộc quốc gia theo khu vực từ dữ liệu thật
    region_country_map = (
        df[["order_region", "order_country"]].dropna().drop_duplicates()
        .groupby("order_region")["order_country"]
        .apply(lambda x: sorted(x.astype(str).unique().tolist()))
        .to_dict()
    )
    col_a, col_b = st.columns(2)
    with col_a:
        shipping_mode = st.selectbox("Phương thức vận chuyển", modes_new)
        order_region = st.selectbox("Khu vực", regions_new)
        order_country = st.selectbox(
            "Quốc gia", options=region_country_map.get(order_region, []),
            format_func=lambda value: COUNTRY_DISPLAY.get(value, value),
            key=f"country_{order_region}"
        )
    with col_b:
        scheduled_days = st.number_input(
            "Số ngày vận chuyển dự kiến", min_value=0, max_value=30, value=3, step=1
        )
        order_month = st.selectbox("Tháng đặt hàng", list(range(1, 13)))
    with st.form("predict_new_order"):
        submitted = st.form_submit_button("Dự báo nguy cơ giao trễ", type="primary")

    if submitted:
        try:
            spark_session, pipeline = load_spark_pipeline(str(model_dir))
            from pyspark.sql.types import StructType, StructField, StringType, IntegerType
            schema = StructType([
                StructField("shipping_mode", StringType(), False),
                StructField("order_region", StringType(), False),
                StructField("order_country", StringType(), False),
                StructField("scheduled_shipping_days", IntegerType(), False),
                StructField("order_month", IntegerType(), False),
            ])
            row = [(shipping_mode, order_region, order_country, int(scheduled_days), int(order_month))]
            new_order = spark_session.createDataFrame(row, schema)
            prediction_row = pipeline.transform(new_order).select("prediction", "probability").first()
            late = int(prediction_row["prediction"]) == 1
            late_probability = float(prediction_row["probability"][1])
            if late:
                st.error("Kết quả: Có nguy cơ giao trễ")
            else:
                st.success("Kết quả: Dự báo không giao trễ")
            st.metric("Xác suất dự báo giao trễ", f"{late_probability:.2%}")
            st.caption("Đây là ước lượng của mô hình, không phải cam kết về thời gian giao hàng thực tế.")
        except Exception as exc:
            st.error("Không thể chạy dự báo. Kiểm tra môi trường PySpark/Java và mô hình đã lưu.")
            st.exception(exc)

# ==================== DỰ BÁO ML ====================
elif page == "Dự báo ML":
    st.subheader("Đánh giá mô hình Random Forest")
    st.caption(
        "Các chỉ số được đánh giá trên tập Test cố định, "
        "không thay đổi theo bộ lọc phân tích."
    )

    y_true = pred["label"].astype(int)
    y_pred = pred["prediction"].astype(int)
    probabilities = pred["probability_late"]

    accuracy = accuracy_score(y_true, y_pred)
    weighted_f1 = f1_score(
        y_true, y_pred, average="weighted"
    )
    fpr, tpr, _ = roc_curve(y_true, probabilities)
    roc_auc = auc(fpr, tpr)

    c1, c2, c3, c4 = st.columns(4)

    with c1:
        kpi("ACCURACY", f"{accuracy:.2%}")
    with c2:
        kpi("WEIGHTED F1", f"{weighted_f1:.4f}")
    with c3:
        kpi("ROC-AUC", f"{roc_auc:.4f}", color=BLUE)
    with c4:
        kpi("TẬP TEST", f"{len(pred):,}", "Số đơn đánh giá")

    c1, c2 = st.columns(2, gap="medium")

    with c1:
        cm = confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        )
        fig = px.imshow(
            cm,
            x=["Không trễ", "Giao trễ"],
            y=["Không trễ", "Giao trễ"],
            text_auto=True,
            color_continuous_scale="Blues",
            title="Ma trận nhầm lẫn"
        )
        fig.update_layout(coloraxis_showscale=False)
        chart(fig, 400)

    with c2:
        fig = go.Figure()
        fig.add_scatter(
            x=fpr, y=tpr,
            mode="lines",
            name=f"ROC-AUC = {roc_auc:.4f}",
            line=dict(color=BLUE, width=3)
        )
        fig.add_scatter(
            x=[0, 1], y=[0, 1],
            mode="lines",
            name="Ngẫu nhiên",
            line=dict(color=MUTED, dash="dash")
        )
        fig.update_layout(
            title="Đường cong ROC",
            xaxis_title="False Positive Rate",
            yaxis_title="True Positive Rate"
        )
        chart(fig, 400)

    comparison = pd.DataFrame({
        "Mô hình": ["RF 12 đặc trưng", "RF 5 đặc trưng"],
        "Accuracy": [0.6948, 0.6949],
        "Weighted F1": [0.6929, 0.6930],
        "ROC-AUC": [0.7309, 0.7345]
    })

    long_df = comparison.melt(
        id_vars="Mô hình",
        var_name="Chỉ số",
        value_name="Giá trị"
    )

    fig = px.bar(
        long_df,
        x="Chỉ số",
        y="Giá trị",
        color="Mô hình",
        barmode="group",
        text="Giá trị",
        title="So sánh Random Forest 12 và 5 đặc trưng",
        color_discrete_map={
            "RF 12 đặc trưng": "#94A3B8",
            "RF 5 đặc trưng": BLUE
        }
    )
    fig.update_traces(texttemplate="%{text:.4f}")
    chart(fig, 390)

    insight(
        "Mô hình Random Forest 5 đặc trưng đạt "
        f"<b>Accuracy {accuracy:.2%}</b>, "
        f"<b>Weighted F1 {weighted_f1:.4f}</b> và "
        f"<b>ROC-AUC {roc_auc:.4f}</b> trên tập Test."
    )

st.divider()
st.caption(
    "GROUP 6  |  BIG DATA ANALYTICS  |  DATACO SMART SUPPLY CHAIN"
)
'''

app_file = os.path.join(dashboard_path, "app.py")

# Kiểm tra cú pháp trước khi lưu
compile(app_code, app_file, "exec")

with open(app_file, "w", encoding="utf-8") as f:
    f.write(app_code)

print("Đã tạo Dashboard chuyên nghiệp thành công!")
print("Đường dẫn:", app_file)

Đã tạo Dashboard chuyên nghiệp thành công!
Đường dẫn: /content/mydrive_new/MyDrive/BigData_Delivery_Prediction/dashboard/app.py


### 9.4. Kiểm tra các tệp Dashboard

In [5]:
import os

required_files = [
    "data_analysis.csv",
    "prediction_test.csv",
    "app.py"
]

for name in required_files:
    path = os.path.join(dashboard_path, name)
    print(f"{name}: {'OK' if os.path.isfile(path) else 'THIẾU'}")

with open(os.path.join(dashboard_path, "app.py"), encoding="utf-8") as f:
    code = f.read()

print("Giao diện mới:", "DELIVERY INTELLIGENCE" in code)

data_analysis.csv: OK
prediction_test.csv: OK
app.py: OK
Giao diện mới: False


### 9.5. Khởi chạy và kiểm tra Dashboard

In [6]:
import subprocess
import time
from google.colab import output

# Dừng Streamlit cũ
subprocess.run(
    ["pkill", "-f", "streamlit run"],
    capture_output=True
)

# Khởi chạy Streamlit mới
log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run",
        dashboard_path + "/app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false"
    ],
    cwd=dashboard_path,
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Streamlit đang chạy:", process.poll() is None)

output.serve_kernel_port_as_window(8501)

Streamlit đang chạy: True
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## PHẦN 10. KẾT LUẬN

Nhóm đã hoàn thành xử lý và tích hợp dữ liệu DataCo bằng PySpark, phân tích tình trạng giao hàng bằng Spark SQL và xây dựng mô hình Random Forest để dự báo trễ hạn giao hàng.

Kết quả phân tích cho thấy tỷ lệ giao hàng trễ là 54,82%. Mô hình Random Forest đạt Accuracy 69,49% và ROC-AUC 0,7345 trên tập Test.

Trong quá trình thực nghiệm, nhóm đã thử bổ sung đặc trưng, tối ưu mô hình và tích hợp thêm dữ liệu thời tiết. Tuy nhiên, các phương án này chưa cải thiện đáng kể hiệu quả dự báo nên nhóm lựa chọn mô hình Random Forest với 5 đặc trưng.

Ngoài ra, nhóm đã xây dựng Dashboard Streamlit để trực quan hóa dữ liệu và kết quả dự báo.

**Hạn chế:** Độ chính xác của mô hình còn ở mức trung bình, các đặc trưng hiện có chưa phản ánh đầy đủ những yếu tố ảnh hưởng đến quá trình giao hàng.

**Hướng phát triển:** Nghiên cứu bổ sung dữ liệu vận hành thực tế như tình trạng kho hàng, đơn vị vận chuyển và quá trình xử lý đơn hàng nhằm nâng cao hiệu quả dự báo.